# **ppkt2synergy Workflow**

This notebook demonstrates a complete workflow for analyzing phenopacket data using the `ppkt2synergy` package. It covers data loading, feature matrix construction, and downstream analysis including correlation and synergy detection between phenotypic features.

---

## Overview

The `ppkt2synergy` package is designed to transform phenopacket data into a structured representation suitable for statistical analysis.

At a high level, the workflow consists of:

1. Parsing phenopacket data
2. Constructing binary feature matrices for HPO terms and diseases
3. Performing pairwise **correlation analysis**
4. Quantifying higher-order relationships via **synergy analysis**
5. Visualizing the results

The goal is to uncover not only individual phenotype–phenotype associations, but also **interactions between phenotypic features** that may carry additional biological or diagnostic significance.

## Install Dependencies

To get started, install the required libraries:

```bash
pip install ppkt2synergy gpsea
```

In [1]:
from ppktstore.registry import configure_phenopacket_registry
from ppkt2synergy import PhenotypeDatasetBuilder, HPOCorrelationAnalyzer, SynergyAnalyzer

### **Loading phenopackets**

In [2]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store() as ps:
        phenopackets = [
            p for p in ps.iter_cohort_phenopackets("FBN1")
        ]

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 143 phenopackets


### **Building dataset for analysis** 

In [3]:
dataset = PhenotypeDatasetBuilder(phenopackets).build()

Individuals Processed: 100%|██████████| 143/143 [00:00<00:00, 444.81 individuals/s]


### **Correlation Analysis Between HPO Terms**

This section demonstrates computing pairwise correlations between HPO terms in the cohort using the `HPOStatisticsAnalyzer` class.


In [4]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)

results = analyzer.compute_correlation_matrix()

results.results_table.head()

Calculating pairwise correlation: 100%|██████████| 728/728 [00:01<00:00, 467.25it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids
467,HP:0004322,Short stature,HP:0004942,Aortic aneurysm,-0.827131,2.049015e-14,9.773799e-12,7,34,33,0,74,5,10756346;11175294;12203992;20375004;21683322
50,HP:0000098,Tall stature,HP:0004942,Aortic aneurysm,0.796637,1.940118e-13,4.627183e-11,45,7,1,26,79,13,10756346;11175294;12203992;20375004;20979188;2...
48,HP:0000098,Tall stature,HP:0004322,Short stature,-0.685681,5.424742e-12,8.625340e-10,16,33,37,0,86,5,10756346;11175294;12203992;20375004;21683322
44,HP:0000098,Tall stature,HP:0002616,Aortic root aneurysm,0.646903,1.766457e-11,2.106500e-09,70,7,10,26,113,18,10756346;11175294;12203992;20375004;20979188;2...
439,HP:0002616,Aortic root aneurysm,HP:0004322,Short stature,-0.634615,2.474657e-10,2.360823e-08,19,33,33,0,85,5,10756346;11175294;12203992;20375004;21683322


#### Saving the results

In [5]:
results.save_correlation_results(
    output_file="correlation_results_FBN1.csv"
)

#### Ploting the heatmap

In [6]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort FBN1",
) 

#### Saving the heatmap

In [7]:
results.save_correlation_heatmap()

### **Synergy Analysis Between HPO Terms and Target Variables**

This section demonstrates computation of **pairwise synergy** between HPO terms with respect to a target variable (e.g., disease status or variant effect) using the `SynergyAnalyzer` class.

In [8]:
diseases_df, sex_df, genes_df, variant_effects_df = dataset.describe_conditions()

diseases_df

,label,n_individuals
disease_id,,
OMIM:154700,Marfan syndrome,50 (35.0%)
OMIM:129600,"Ectopia lentis, familial",44 (30.8%)
OMIM:614185,Geleophysic dysplasia 2,19 (13.3%)
OMIM:102370,Acromicric dysplasia,13 (9.1%)
OMIM:616914,Marfan lipodystrophy syndrome,9 (6.3%)
OMIM:184900,Stiff skin syndrome,8 (5.6%)


In [9]:
sex_df

,n_individuals
sex,
female,57 (39.9%)
male,54 (37.8%)
unknown,32 (22.4%)


In [10]:
genes_df

,n_individuals
gene_symbol,
FBN1,143 (100.0%)


In [11]:
variant_effects_df

transcript_id,NM_000138.5,NM_001406716.1,NM_001406717.1
variant_effect,,,
MISSENSE_VARIANT,119 (78.8%),119 (78.8%),8 (88.9%)
SPLICE_REGION_VARIANT,6 (4.0%),6 (4.0%),1 (11.1%)
FRAMESHIFT_VARIANT,7 (4.6%),7 (4.6%),0 (0.0%)
SPLICE_DONOR_VARIANT,6 (4.0%),6 (4.0%),0 (0.0%)
STOP_GAINED,4 (2.6%),4 (2.6%),0 (0.0%)
INFRAME_INSERTION,1 (0.7%),1 (0.7%),0 (0.0%)
SPLICE_ACCEPTOR_VARIANT,2 (1.3%),2 (1.3%),0 (0.0%)
SPLICE_DONOR_REGION_VARIANT,1 (0.7%),1 (0.7%),0 (0.0%)
INTRON_VARIANT,2 (1.3%),2 (1.3%),0 (0.0%)


In [12]:
synergy_analyzer = SynergyAnalyzer(dataset=dataset)

In [13]:
from ppkt2synergy import has_disease
condition_disease = dataset.get_condition(has_disease("OMIM:154700"), name="disease:Marfan syndrome")

In [14]:
results_disease = synergy_analyzer.compute_synergy_matrix(
    condition=condition_disease
)
results_disease.results_table.head()

Calculating pairwise synergy: 100%|██████████| 606/606 [01:10<00:00,  8.61it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,adj_p_value,n(A:E/B:E)_y0,n(A:E/B:O)_y0,n(A:O/B:E)_y0,n(A:O/B:O)_y0,N_y0,n(A:E/B:E)_y1,n(A:E/B:O)_y1,n(A:O/B:E)_y1,n(A:O/B:O)_y1,N_y1,n_individuals,n_pmids,pmids
179,HP:0001519,Disproportionate tall stature,HP:0004942,Aortic aneurysm,-0.287324,0.0001,0.000526,45,3,0,0,48,0,17,0,17,34,82,12,10756346;11175294;12203992;20375004;20979188;2...
178,HP:0001519,Disproportionate tall stature,HP:0004322,Short stature,-0.182472,0.0001,0.000526,7,33,0,0,40,29,0,21,0,50,90,5,10756346;11175294;12203992;20375004;21683322
157,HP:0001382,Joint hypermobility,HP:0001659,Aortic regurgitation,0.038321,0.0001,0.000526,29,2,0,0,31,3,2,5,0,10,41,2,11175294;21683322
225,HP:0002650,Scoliosis,HP:0030053,Stiff skin,-0.110839,0.0001,0.000526,0,8,0,0,8,27,0,23,0,50,58,4,10756346;11175294;12203992;20375004
163,HP:0001382,Joint hypermobility,HP:0003510,Severe short stature,-0.133192,0.0001,0.000526,11,18,0,0,29,26,0,22,0,48,77,4,10756346;11175294;12203992;21683322


In [15]:
results_disease.save_synergy_results(
    output_file='synergy_matrix_FBN1.csv',
)

In [16]:
results_disease.plot_synergy_heatmap()

In [17]:
results_disease.save_synergy_heatmap(
    output_file="synergy_heatmap_FBN1.html"
)